***Enunciado***

Muchos objetos astrofísicos fascinantes (núcleos galácticos activos, hipernovas, binarias de rayos X) pueden explicarse con los fenómenos que ocurren en los denominados discos de acreción alrededor de agujeros negros. En este proyecto se usa lo visto en el curso de Relatividad y Gravitación para estudiar ese problema.


El objetivo de este trabajo es desarrollar simulaciones y cálculos útiles que permitan comparar la emisión local del disco con la observada desde distintos ángulos, usando los efectos relativistas de: 

- factor de Lorentz y velocidad relativista,
- beaming relativista y aberración de la luz,
- desplazamiento Doppler,
- corrimiento al rojo gravitacional en el potencial de Schwarzschild.

En el notebook se modela un disco de acreción con dinámica newtoniana para las partículas del gas, y se incorporan correcciones relativistas para el flujo observado y la longitud de onda observada. Las ideas fundamentales se toman de los notebooks de clase.

***Criterios usados en este notebook***

- Simulación de la dinámica orbital de partículas en un disco kepleriano.
- Cálculo de la emisión térmica del disco con perfil de temperatura tipo Shakura-Sunyaev.
- Correcciones relativistas de beaming y desplazamiento espectral.
- Visualización de mapas de flujo y perfiles radiales para diferentes inclinaciones.


In [35]:
import numpy as np  # Importar NumPy para manejar arrays y operaciones matemáticas
import matplotlib.pyplot as plt  # Importar Matplotlib para crear gráficos
import matplotlib.gridspec as gridspec  # Para organizar subgráficos en una cuadrícula
from matplotlib.colors import LinearSegmentedColormap, Normalize  # Para mapas de colores personalizados
from matplotlib.patches import Circle  # Para dibujar círculos y flechas en gráficos
from scipy.integrate import odeint  # Para integrar ecuaciones diferenciales ordinarias

# UNIDADES DE AÑO LUZ (Light-Year Units)
c_SI = 299792458  # Velocidad de la luz en metros por segundo (valor SI)
c = 1  # Velocidad de la luz en unidades relativistas (adimensional en años-luz/año)
UT = 365.25 * 86400  # Segundos en un año (365.25 días * 86400 segundos/día)
UL = c_SI * UT  # Unidad de longitud = año luz en metros
UV = UL / UT  # Unidad de velocidad = años-luz por año en metros por segundo
UA = UL / UT**2  # Unidad de aceleración = años-luz por año cuadrado

# CONSTANTES Y PARÁMETROS GLOBALES (en unidades de año luz)
G_SI = 6.674e-11  # Constante gravitacional en m³ kg⁻¹ s⁻² (valor SI)
M_sun_SI = 1.989e30  # Masa del Sol en kilogramos (valor SI)

# Convertir G a unidades de año luz: años-luz³ / (masa solar · año²)
G = G_SI * M_sun_SI * UT**2 / UL**3

M_sun = 1.0  # Masa del Sol en las nuevas unidades (1 masa solar)
M_BH = 10 * M_sun  # Masa del agujero negro en 10 masas solares

# Radio de Schwarzschild en años-luz
# Rs = 2 * G * M / c², con c = 1 en unidades relativistas
R_s = 2 * G * M_BH / c**2
R_isco = 3 * R_s  # Radio del innermost stable circular orbit (ISCO)

# Convertir a metros para visualización
R_s_m = R_s * UL  # Radio de Schwarzschild en metros
R_isco_m = R_isco * UL  # ISCO en metros

# ABERRACIÓN RELATIVISTA POR EL MOVIMIENTO DE LA TIERRA
v_tierra = np.array([0, 2.978e4, 0])  # Velocidad orbital de la Tierra en m/s (dirección y)
beta_vec = -v_tierra / c_SI  # Vector beta = velocidad / c, negativo para dirección opuesta
beta2 = np.sum(beta_vec**2)  # Norma al cuadrado del vector beta
gamma_earth = 1 / np.sqrt(1 - beta2)  # Factor de Lorentz para la Tierra
nprima_ecl = np.array([0, 0, 1])  # Dirección eclíptica (asumir eje z)
dot = beta_vec @ nprima_ecl  # Producto punto entre beta_vec y dirección
beta_earth = np.sqrt(beta2)  # Norma de beta
cos_theta_earth = dot / beta_earth  # Coseno del ángulo entre velocidad y dirección
z_earth = gamma_earth * (1 - beta_earth * cos_theta_earth) - 1  # Redshift por movimiento de la Tierra
factor_earth = (1 + z_earth)**3  # Factor de corrección para flujo (1+z)^3



# Teoría: Módulo 1 - Dinámica Newtoniana

## Órbitas Keplerianas

En mecánica clásica, una partícula de masa $m$ en órbita circular alrededor de una masa central $M$ satisface el balance entre la fuerza gravitacional y la aceleración centrípeta:

$$\frac{GM}{r^2} = \frac{v_c^2}{r}$$

Despejando la velocidad circular:

$$v_c(r) = \sqrt{\frac{GM}{r}}$$

El período orbital viene dado por la tercera ley de Kepler:

$$T(r) = \frac{2\pi r}{v_c(r)} = 2\pi\sqrt{\frac{r^3}{GM}}$$

## Ecuaciones de Movimiento 2D

Para simular partículas en un disco, usamos las ecuaciones newtonianas en 2D con un término de fricción débil que simula la acreción:

$$\begin{cases}
\frac{dx}{dt} = v_x \\
\frac{dy}{dt} = v_y \\
\frac{dv_x}{dt} = -\frac{GMx}{r^3} + f_{\text{drag}} \cdot v_x \\
\frac{dv_y}{dt} = -\frac{GMy}{r^3} + f_{\text{drag}} \cdot v_y
\end{cases}$$

donde $r = \sqrt{x^2 + y^2}$ es la distancia radial y $f_{\text{drag}} \approx -5 \times 10^{-5}$ es un coeficiente de fricción.

## Condiciones Iniciales

Las partículas se distribuyen uniformemente entre:
- Radio mínimo: $r_{\min} = 1.05 \times R_{\text{ISCO}}$
- Radio máximo: $r_{\max} = 20 R_s$

Con pequeñas excentricidades iniciales ($e \approx 0.05$) para quebrar la simetría perfecta.

In [36]:
# MÓDULO 1: DINÁMICA NEWTONIANA
# ═══════════════════════════════════════════════════════════════════════════
# TEORÍA:
#   • Órbitas Keplerianas: v_c = √(GM/r)
#   • Período orbital: T = 2π√(r³/GM) — Tercera ley de Kepler
#   • Ecuaciones de movimiento 2D con fricción débil
#   • Simulación de N partículas en disco de acreción
# OBJETIVO: Calcular trayectorias de partículas bajo gravedad newtoniana
# ═══════════════════════════════════════════════════════════════════════════


def Vel_cir(r):
    """Calcular la velocidad circular kepleriana en el radio r."""
    return np.sqrt(G * M_BH / r)  # Fórmula: v = sqrt(G M / r)

def T_orb(r):
    """Calcular el período orbital kepleriano en el radio r."""
    return 2 * np.pi * r / Vel_cir(r)  # Fórmula: T = 2π r / v

def Equ_mov(state, t):
    """
    Sistema de ecuaciones diferenciales ordinarias para movimiento gravitacional 2D.
    state = [x, y, vx, vy] representa posición y velocidad.
    """
    x, y, vx, vy = state  # Desempaquetar el estado
    r = np.sqrt(x**2 + y**2)  # Calcular la distancia radial
    # Añadir fuerza de fricción débil para simular acreción
    drag = -5e-5  # Coeficiente de arrastre
    ax = -G * M_BH * x / r**3 + drag * vx  # Aceleración en x: gravedad + fricción
    ay = -G * M_BH * y / r**3 + drag * vy  # Aceleración en y: gravedad + fricción
    return [vx, vy, ax, ay]  # Retornar derivadas: [dx/dt, dy/dt, dvx/dt, dvy/dt]

def Simulacion(n_particles=2000, t_max=None, n_steps=1500):
    """
    Simular N partículas con condiciones iniciales casi circulares.
    Las partículas se distribuyen entre ISCO y 20 R_s.
    """
    np.random.seed(42)  # Fijar semilla para reproducibilidad

    # Radios distribuidos uniformemente
    r_min = R_isco * 1.05  # Radio mínimo ligeramente mayor que ISCO
    r_max = 20 * R_s  # Radio máximo 20 veces Rs
    radii = np.random.uniform(r_min, r_max, n_particles)  # Generar radios aleatorios
    angles = np.random.uniform(0, 2*np.pi, n_particles)  # Ángulos aleatorios

    # Perturbación radial pequeña (excentricidad ≈ 0.05)
    ecc = np.random.uniform(0, 0.08, n_particles)  # Excentricidades aleatorias

    particles = []  # Lista para almacenar las trayectorias
    for i in range(n_particles):
        r = radii[i]  # Radio de la partícula i
        phi = angles[i]  # Ángulo inicial
        v_c = Vel_cir(r) * (1 + ecc[i] * np.random.choice([-1, 1]))  # Velocidad con perturbación

        x = r * np.cos(phi)  # Posición inicial x
        y = r * np.sin(phi)  # Posición inicial y
        vx = -v_c * np.sin(phi)  # Velocidad inicial vx (tangencial)
        vy = v_c * np.cos(phi)  # Velocidad inicial vy (tangencial)

        T = T_orb(r)  # Período orbital
        if t_max is None:
            t_max_i = 3 * T  # Tiempo máximo 3 períodos si no especificado
        else:
            t_max_i = t_max

        t = np.linspace(0, t_max_i, n_steps)  # Array de tiempos
        sol = odeint(Equ_mov, [x, y, vx, vy], t,
                     rtol=1e-9, atol=1e-12)  # Integrar EDOs

        # Eliminar partículas que caen al agujero negro
        r_traj = np.sqrt(sol[:, 0]**2 + sol[:, 1]**2)  # Radio a lo largo de la trayectoria
        mask = r_traj > R_s * 1.1  # Máscara para radios > 1.1 Rs
        if mask.sum() > 10:  # Si hay suficientes puntos
            particles.append({
                'x': sol[mask, 0],  # Posiciones x filtradas
                'y': sol[mask, 1],  # Posiciones y filtradas
                'vx': sol[mask, 2],  # Velocidades vx filtradas
                'vy': sol[mask, 3],  # Velocidades vy filtradas
                'r0': r,  # Radio inicial
                'r_traj': r_traj[mask],  # Radios filtrados
            })

    return particles  # Retornar lista de partículas


**Teoría: Módulo 2 - Efectos Relativistas Cinemáticos**

**Factor de Lorentz**

En relatividad especial, las velocidades se relacionan con el factor de Lorentz:

$$\gamma = \frac{1}{\sqrt{1 - \beta^2}}, \quad \text{donde} \quad \beta = \frac{v}{c}$$

Este factor es fundamental para describir la dilatación del tiempo y la contracción del espacio.

**Beaming Relativista**

Cuando una fuente se mueve hacia un observador, la intensidad observada se amplifica por un factor $\delta^3$, donde:

$$\delta = \gamma(1 - \beta \cos\theta)$$

es el factor Doppler relativista. Así, el flujo observado es:

$$F_{\text{obs}} = \frac{F_{\text{rest}}}{\delta^3}$$

Este efecto es crucial en discos de acreción: el lado que se acerca al observador brilla mucho más.

**Desplazamiento Doppler**

La longitud de onda observada experimenta un desplazamiento:

$$\lambda_{\text{obs}} = \lambda_{\text{rest}} \cdot \delta = \lambda_{\text{rest}} \cdot \gamma(1 - \beta\cos\theta)$$

- Si la fuente se acerca: $\delta < 1$ → desplazamiento al azul (blueshift)
- Si la fuente se aleja: $\delta > 1$ → desplazamiento al rojo (redshift)

In [37]:
# MÓDULO 2: RELATIVIDAD ESPECIAL (EFECTOS RELATIVISTAS CINEMÁTICOS)
# ═══════════════════════════════════════════════════════════════════════════
# TEORÍA:
#   • Factor de Lorentz: γ = 1/√(1 - β²) donde β = v/c
#   • Beaming relativista: F_obs = F_rest / δ³ donde δ = γ(1 - β cos θ)
#   • Desplazamiento Doppler: λ_obs = λ_rest × δ
#   • Redshift gravitacional: λ_obs = λ_emit / √(1 - R_s/r)
# OBJETIVO: Aplicar correcciones relativistas a velocidades y flujos
# ═══════════════════════════════════════════════════════════════════════════

def Factor_L(v):
    """Calcular el factor de Lorentz γ = 1/√(1 - v²/c²)."""
    beta = np.clip(np.abs(v) / c, 0, 0.9999)  # Calcular β = |v| / c, limitado entre 0 y 0.9999 para evitar errores numéricos
    return 1.0 / np.sqrt(1 - beta**2)  # Retornar γ usando la fórmula relativista estándar

def Beaming_r(flux_rest, gamma, cos_theta):
    """
    Calcular el beaming relativista.
    F_obs = F_rest / [γ(1 - β cos θ)]^3
    θ: ángulo entre velocidad y línea de visión.
    """
    beta = np.sqrt(1 - 1/gamma**2)  # Calcular β = √(1 - 1/γ²) desde el factor de Lorentz
    doppler = gamma * (1 - beta * cos_theta)  # Calcular el factor Doppler relativista
    doppler = np.clip(doppler, 1e-6, 1e6)  # Limitar el factor Doppler para estabilidad numérica
    return flux_rest / doppler**3  # Retornar el flujo observado con beaming relativista

def Doppler(lambda_rest, gamma, cos_theta):
    """
    Calcular el desplazamiento Doppler relativista.
    λ_obs = λ_rest · γ(1 - β cos θ)
    """
    beta = np.sqrt(1 - 1/gamma**2)  # Calcular β desde el factor de Lorentz γ
    return lambda_rest * gamma * (1 - beta * cos_theta)  # Retornar la longitud de onda observada con desplazamiento Doppler

def Redshitf(lambda_emit, r):
    """
    Calcular el corrimiento al rojo gravitacional.
    λ_obs = λ_emit / √(1 - R_s/r)
    """
    factor = np.sqrt(1 - R_s / np.clip(r, R_s*1.01, 1e20))  # Calcular el factor gravitacional, limitando r para evitar división por cero
    return lambda_emit / factor  # Retornar la longitud de onda con corrimiento al rojo gravitacional

def T_disco(r):
    """
    Calcular el perfil de temperatura del disco (aproximación Shakura-Sunyaev).
    T(r) ∝ r^{-3/4} · (1 - √(R_isco/r))^{1/4}
    """
    if r <= R_isco:
        return 0  # Retornar temperatura cero si está dentro del radio ISCO
    T_max = 1e7  # Temperatura máxima en Kelvin
    x = R_isco / r  # Calcular la razón R_isco / r
    return T_max * (r / R_isco)**(-3/4) * max((1 - np.sqrt(x)), 0)**0.25  # Retornar la temperatura usando el perfil de Shakura-Sunyaev

Módulo 3: Radiación Térmica y Redshift Gravitacional1. Corrimiento Gravitacional (Redshift de Schwarzschild)En las cercanías de un objeto masivo como un agujero negro, el tiempo transcurre de forma distinta debido a la curvatura del espacio-tiempo definida por la métrica de Schwarzschild:$$ds^2 = -\left(1 - \frac{R_s}{r}\right) c^2 dt^2 + \frac{dr^2}{1 - R_s/r} + r^2 d\Omega^2$$Efecto en la luzCuando un fotón escapa del pozo gravitacional, pierde energía. Como la energía de un fotón es inversamente proporcional a su longitud de onda ($E = hc/\lambda$), la longitud de onda aumenta (se desplaza al rojo):$$\lambda_{\text{obs}} = \frac{\lambda_{\text{emit}}}{\sqrt{1 - \frac{R_s}{r}}}$$Interpretación: Si $r \to R_s$, el denominador tiende a cero y $\lambda_{\text{obs}}$ tiende a infinito. Esto significa que la luz se vuelve invisible para el observador lejano justo en el horizonte de eventos.2. Perfil de Temperatura (Shakura-Sunyaev)Un disco de acreción no tiene una temperatura uniforme; es mucho más caliente en el centro debido a la liberación de energía potencial gravitatoria y la fricción viscosa. El perfil de temperatura para un disco en estado estacionario es:$$T(r) = T_0 \left(\frac{R_{\text{ISCO}}}{r}\right)^{3/4} \left(1 - \sqrt{\frac{R_{\text{ISCO}}}{r}}\right)^{1/4}$$Componentes de la fórmula:$T_0 \sim 10^7$ K: Temperatura de escala (típica en AGNs o binarias de rayos X).$(R/r)^{3/4}$: Representa la disipación de energía por viscosidad.Término de borde $(1 - \sqrt{R_i/r})^{1/4}$: Asegura que la temperatura caiga a cero en la ISCO (Inner Most Stable Circular Orbit), donde el gas deja de orbitar y cae directamente al agujero negro.3. Radiación de Cuerpo Negro y Ley de Wien (Corregida)Cada sección del disco emite radiación térmica. La distribución de esta energía sigue la Ley de Planck:$$B_\lambda(T) = \frac{2hc^2}{\lambda^5} \frac{1}{e^{\frac{hc}{\lambda k_B T}} - 1}$$Corrección de la Ley de WienLa relación entre la temperatura de una región del disco y la longitud de onda donde emite su máxima intensidad es:$$\lambda_{\text{pico}} = \frac{b}{T}$$Constante de desplazamiento de Wien ($b$):$$b \approx 2.8977 \times 10^{-3} \text{ m}\cdot\text{K}$$

In [38]:
# ═══════════════════════════════════════════════════════════════════════════
# MÓDULO 3: RADIACIÓN TÉRMICA Y MAPAS DE EMISIÓN
# ═══════════════════════════════════════════════════════════════════════════
# TEORÍA:
#   • Perfil de temperatura Shakura-Sunyaev: T(r) ∝ (r/r_isco)^(-3/4) × (1-√(r_isco/r))^(1/4)
#   • Ley de Planck: B_λ = 2hc²/[λ⁵(e^(hc/λk_B T) - 1)]
#   • Ley de Wien: λ_pico = b/T (aproximación para pico de emisión)
#   • Combinación de beaming relativista + redshift gravitacional
# OBJETIVO: Generar mapas 2D de flujo y longitud de onda observados
# ═══════════════════════════════════════════════════════════════════════════

def Mapa_Disco(N_r=80, N_phi=200, obs_angle_deg=75):
    """
    Para un disco kepleriano, calcula:
    - Flujo observado (con beaming)
    - Longitud de onda observada (Doppler + gravit.)
    - Temperatura local
    Ángulo de observación medido desde el eje polar (inclinación).
    """
    obs_inc = np.radians(obs_angle_deg)  # Convertir ángulo de observación de grados a radianes

    r_arr   = np.linspace(R_isco * 1.05, 18 * R_s, N_r)  # Array de radios desde ISCO hasta 18 Rs
    phi_arr = np.linspace(0, 2*np.pi, N_phi, endpoint=False)  # Array de ángulos azimutales sin duplicar el punto final
    PHI, R  = np.meshgrid(phi_arr, r_arr)  # Crear malla 2D para phi y r

    # Velocidad kepleriana (en unidades donde c=1)
    V_kep = np.sqrt(G * M_BH / R)  # Calcular velocidad kepleriana usando ley de Kepler
    # Componente de la velocidad a lo largo de la línea de visión
    # v_los = v_phi · sin(phi) · sin(inc)
    V_los = V_kep * np.sin(PHI) * np.sin(obs_inc)  # Componente de velocidad hacia el observador

    GAMMA = Factor_L(V_kep)  # Calcular factor de Lorentz para cada punto (usa c=1 internamente)
    COS_THETA = np.where(V_kep > 0, V_los / V_kep, 0)  # cos θ = sin(φ)·sin(inc)

    # Flujo en reposo ∝ T^4 (cuerpo negro)
    T_mat = np.vectorize(T_disco)(R)  # Calcular temperatura en cada punto de la malla
    F_rest = np.where(T_mat > 0, T_mat**4, 0)  # Flujo en reposo proporcional a T^4

    # Flujo observado con beaming
    F_obs = Beaming_r(F_rest, GAMMA, COS_THETA)  # Aplicar beaming relativista

    # Longitud de onda emitida (pico de cuerpo negro, Ley de Wien)
    b_wien = 2.898e-3  # Constante de Wien en metros·Kelvin
    lam_rest = np.where(T_mat > 0, b_wien / T_mat * 1e9, 0)  # Longitud de onda en reposo en nm

    # Longitud de onda observada (Doppler + redshift gravitacional)
    # β = V_kep / c = V_kep  (c=1 en estas unidades)
    # Factor Doppler: δ = γ(1 - β cos θ)
    delta_doppler = GAMMA * (1 - V_kep * COS_THETA)  # Factor Doppler: β = V_kep (c=1)

    # Redshift gravitacional: λ_grav = λ₀ / √(1 - R_s/r)
    grav_redshift = 1.0 / np.sqrt(np.clip(1 - R_s / R, 1e-6, 1))  # Redshift gravitacional (evitar singularidades)

    # Longitud de onda observada combinando ambos efectos
    lam_obs = lam_rest * delta_doppler * grav_redshift  # λ_obs = λ_rest × δ_Doppler × f_gravitacional

    return R, PHI, F_obs, lam_obs, T_mat, V_kep, GAMMA

In [39]:
# ═══════════════════════════════════════════════════════════════════════════
# FUNCIONES DE VISUALIZACIÓN (MÓDULOS 1-3)
# ═══════════════════════════════════════════════════════════════════════════


def make_black_hole_colormap():
    """Colormap disco de acreción: negro → naranja → amarillo → blanco."""
    colors = ['#000000', '#1a0500', '#4d1500', '#a83200',  # Definir colores desde negro hasta naranja
              '#ff6600', '#ffaa00', '#ffdd88', '#ffffff']  # Continuar con amarillo y blanco
    return LinearSegmentedColormap.from_list('accretion', colors, N=512)  # Crear colormap personalizado

def plot_simulation(particles, title="Dinámica Newtoniana del Disco de Acreción"):
    """Figura 1: trayectorias de las partículas, snapshot + trayectorias."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.5),  # Crear figura con dos paneles
                              facecolor='#050508')  # Fondo oscuro
    fig.suptitle(title, color='#f0c060', fontsize=15,  # Título de la figura
                 fontweight='bold', y=0.97)

    for ax in axes:  # Configurar estilo de los ejes
        ax.set_facecolor('#050508')  # Fondo oscuro para ejes
        for sp in ax.spines.values():  # Bordes de los ejes
            sp.set_color('#333344')  # Color gris para bordes
        ax.tick_params(colors='#aaaacc')  # Color de ticks
        ax.xaxis.label.set_color('#aaaacc')  # Color de etiquetas x
        ax.yaxis.label.set_color('#aaaacc')  # Color de etiquetas y

    scale = R_s  # Escala para normalizar posiciones
    cmap_disk = make_black_hole_colormap()  # Colormap para el disco

    # ── Panel izquierdo: snapshot (posición final)
    ax0 = axes[0]  # Primer eje (izquierdo)
    ax0.set_title("Snapshot final", color='#f0c060', fontsize=12)  # Título del panel

    # Color por radio
    snap_x, snap_y, snap_r = [], [], []  # Listas para posiciones finales
    for p in particles:  # Recorrer partículas
        snap_x.append(p['x'][-1] / scale)  # Posición x final normalizada
        snap_y.append(p['y'][-1] / scale)  # Posición y final normalizada
        snap_r.append(p['r_traj'][-1] / scale)  # Radio final normalizado
    snap_x, snap_y, snap_r = map(np.array, [snap_x, snap_y, snap_r])  # Convertir a arrays

    norm_r = Normalize(vmin=snap_r.min(), vmax=snap_r.max())  # Normalización para colormap
    sc = ax0.scatter(snap_x, snap_y, c=snap_r, cmap=cmap_disk,  # Scatter plot coloreado por radio
                     s=4, alpha=0.85, norm=norm_r, zorder=3)

    # Agujero negro
    bh = Circle((0, 0), R_s/scale, color='black', zorder=10)  # Círculo para el agujero negro
    ax0.add_patch(bh)  # Agregar al eje
    isco_circle = Circle((0, 0), R_isco/scale, fill=False,  # Círculo para ISCO
                          edgecolor='#ff4444', linewidth=0.8,
                          linestyle='--', zorder=5, label=f"ISCO = {R_isco/scale:.1f} Rs")
    ax0.add_patch(isco_circle)  # Agregar ISCO

    lim = 22  # Límite de ejes
    ax0.set_xlim(-lim, lim); ax0.set_ylim(-lim, lim)  # Establecer límites
    ax0.set_aspect('equal')  # Aspecto igual
    ax0.set_xlabel("x / Rs"); ax0.set_ylabel("y / Rs")  # Etiquetas
    ax0.legend(loc='upper right', fontsize=9, facecolor='#111122',  # Leyenda
               edgecolor='#334', labelcolor='#ffaaaa')
    cbar = fig.colorbar(sc, ax=ax0, fraction=0.04, pad=0.02)  # Barra de color
    cbar.set_label("r / Rs", color='#aaaacc'); cbar.ax.yaxis.set_tick_params(color='#aaaacc')  # Etiqueta barra
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#aaaacc')  # Color ticks barra

    # ── Panel derecho: trayectorias (muestra 60 partículas)
    ax1 = axes[1]  # Segundo eje (derecho)
    ax1.set_title("Trayectorias de partículas (selección)", color='#f0c060', fontsize=12)  # Título del panel

    shown = particles[::max(1, len(particles)//60)]  # Seleccionar subconjunto de partículas
    for p in shown:  # Recorrer partículas seleccionadas
        r_norm = p['r0'] / (20 * R_s)  # Normalizar radio inicial para colormap
        col = cmap_disk(r_norm)  # Color basado en radio
        ax1.plot(p['x'] / scale, p['y'] / scale,  # Plotear trayectoria
                 color=col, linewidth=0.5, alpha=0.6)

    bh1 = Circle((0, 0), R_s/scale, color='black', zorder=10)  # Agujero negro
    ax1.add_patch(bh1)
    isco1 = Circle((0, 0), R_isco/scale, fill=False,  # ISCO
                   edgecolor='#ff4444', linewidth=0.8, linestyle='--', zorder=5)
    ax1.add_patch(isco1)

    ax1.set_xlim(-lim, lim); ax1.set_ylim(-lim, lim)  # Límites
    ax1.set_aspect('equal')  # Aspecto igual
    ax1.set_xlabel("x / Rs"); ax1.set_ylabel("y / Rs")  # Etiquetas
    plt.tight_layout()  # Ajustar layout


def plot_flux_maps(obs_angles=(15, 45, 75)):
    """Mostrar solo los mapas de flujo observado con beaming
    para cada ángulo de inclinación.
    """
    n_panels = len(obs_angles)
    fig = plt.figure(figsize=(6 * n_panels, 5), facecolor='#050508')
    gs = gridspec.GridSpec(1, n_panels, figure=fig, hspace=0.35, wspace=0.25)
    fig.suptitle("Flujo observado del Disco de Acreción (Beaming)",
                 color='#f0c060', fontsize=15, fontweight='bold', y=0.98)

    cmap_flux = make_black_hole_colormap()

    for col, inc in enumerate(obs_angles):
        R, PHI, F_obs, lam_obs, T_mat, V_kep, GAMMA = Mapa_Disco(obs_angle_deg=inc)
        X = R * np.cos(PHI) / R_s
        Y = R * np.sin(PHI) / R_s

        ax = fig.add_subplot(gs[0, col])
        ax.set_facecolor('#050508')
        F_log = np.log10(np.clip(F_obs, 1, None))
        dr = R[1, 0] - R[0, 0]
        phi_edges = np.linspace(0, 2*np.pi, PHI.shape[1] + 1)
        r_edges = np.linspace(R[0, 0] - dr/2, R[-1, 0] + dr/2, R.shape[0] + 1)
        PHI_edges, R_edges = np.meshgrid(phi_edges, r_edges)
        X_edges = R_edges * np.cos(PHI_edges) / R_s
        Y_edges = R_edges * np.sin(PHI_edges) / R_s
        pc = ax.pcolormesh(X_edges, Y_edges, F_log, cmap=cmap_flux, shading='auto')
        bh = Circle((0, 0), 1.0, color='black', zorder=5)
        ax.add_patch(bh)
        isco = Circle((0, 0), 3.0, fill=False, edgecolor='#ff4444', linewidth=0.7,
                      linestyle='--', zorder=6)
        ax.add_patch(isco)
        ax.set_aspect('equal')
        ax.set_xlim(-19, 19); ax.set_ylim(-19, 19)
        ax.set_title(f"Flujo observado\ninc = {inc}°", color='#f0c060', fontsize=10)
        ax.tick_params(colors='#aaaacc', labelsize=7)
        cb = fig.colorbar(pc, ax=ax, fraction=0.04, pad=0.02)
        cb.set_label("log₁₀(Flujo)", color='#aaaacc', fontsize=8)
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='#aaaacc', fontsize=7)
    plt.show()


def plot_spectral_ratio_maps(obs_angles=(15, 45, 75)):
    """Mostrar solo los mapas de desplazamiento espectral
    λ_obs / λ_rest para cada ángulo de inclinación.
    """
    n_panels = len(obs_angles)
    fig = plt.figure(figsize=(6 * n_panels, 5), facecolor='#050508')
    gs = gridspec.GridSpec(1, n_panels, figure=fig, hspace=0.35, wspace=0.25)
    fig.suptitle("Desplazamiento espectral del Disco de Acreción",
                 color='#f0c060', fontsize=15, fontweight='bold', y=0.98)
    
    cmap_shift = LinearSegmentedColormap.from_list(
        'doppler', ['#0033ff', '#3399ff', '#ffffff', '#ff9933', '#ff2200'], N=512)

    for col, inc in enumerate(obs_angles):
        R, PHI, F_obs, lam_obs, T_mat, V_kep, GAMMA = Mapa_Disco(obs_angle_deg=inc)
        ax = fig.add_subplot(gs[0, col])
        ax.set_facecolor('#050508')

        b_wien = 2.898e-3
        lam_rest_map = np.where(T_mat > 0, b_wien / T_mat * 1e9, np.nan)
        ratio = np.where(lam_rest_map > 0, lam_obs / lam_rest_map, np.nan)

        dr = R[1, 0] - R[0, 0]
        phi_edges = np.linspace(0, 2*np.pi, PHI.shape[1] + 1)
        r_edges = np.linspace(R[0, 0] - dr/2, R[-1, 0] + dr/2, R.shape[0] + 1)
        PHI_edges, R_edges = np.meshgrid(phi_edges, r_edges)
        X_edges = R_edges * np.cos(PHI_edges) / R_s
        Y_edges = R_edges * np.sin(PHI_edges) / R_s
        pc = ax.pcolormesh(X_edges, Y_edges, ratio, cmap=cmap_shift, shading='auto', vmin=0.6, vmax=1.4)
        bh = Circle((0, 0), 1.0, color='black', zorder=5)
        ax.add_patch(bh)
        isco = Circle((0, 0), 3.0, fill=False, edgecolor='#ff4444', linewidth=0.7,
                      linestyle='--', zorder=6)
        ax.add_patch(isco)
        ax.set_aspect('equal')
        ax.set_xlim(-19, 19); ax.set_ylim(-19, 19)
        ax.set_title(f"λ_obs / λ_rest\ninc = {inc}°", color='#f0c060', fontsize=10)
        ax.tick_params(colors='#aaaacc', labelsize=7)
        cb = fig.colorbar(pc, ax=ax, fraction=0.04, pad=0.02)
        cb.set_label("λ_obs / λ_rest", color='#aaaacc', fontsize=8)
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='#aaaacc', fontsize=7)
    plt.show()


def plot_radial_flux_profiles(obs_angles=(15, 45, 75)):
    """Mostrar solo los perfiles radiales de flujo integrado
    para cada ángulo de inclinación.
    """
    n_panels = len(obs_angles)
    fig = plt.figure(figsize=(6 * n_panels, 5), facecolor='#050508')
    gs = gridspec.GridSpec(1, n_panels, figure=fig, hspace=0.35, wspace=0.25)
    fig.suptitle("Perfil radial de flujo integrado sobre φ",
                 color='#f0c060', fontsize=15, fontweight='bold', y=0.98)

    for col, inc in enumerate(obs_angles):
        R, PHI, F_obs, lam_obs, T_mat, V_kep, GAMMA = Mapa_Disco(obs_angle_deg=inc)
        r_vals = R[:, 0] / R_s
        flux_integrated = np.nansum(F_obs, axis=1)
        flux_rest_int = np.nansum(T_mat**4, axis=1)
        fn = flux_integrated / max(flux_integrated.max(), 1)
        fr = flux_rest_int / max(flux_rest_int.max(), 1)

        ax = fig.add_subplot(gs[0, col])
        ax.set_facecolor('#050508')
        for sp in ax.spines.values():
            sp.set_color('#333344')
        ax.tick_params(colors='#aaaacc', labelsize=8)
        ax.fill_between(r_vals, fn, alpha=0.3, color='#ff8800')
        ax.plot(r_vals, fn, color='#ffcc44', linewidth=1.5, label='F_obs (beaming)')
        ax.plot(r_vals, fr, color='#4488ff', linewidth=1.2, linestyle='--', label='F_rest (sin beaming)')
        ax.axvline(3, color='#ff4444', linewidth=0.8, linestyle='--', label='ISCO')
        ax.set_xlabel("r / Rs", color='#aaaacc', fontsize=9)
        ax.set_ylabel("Flujo norm.", color='#aaaacc', fontsize=9)
        ax.set_title(f"Perfil radial inc = {inc}°", color='#f0c060', fontsize=10)
        ax.legend(fontsize=7.5, facecolor='#111122', edgecolor='#334', labelcolor='#ddddff')
        ax.set_xlim(r_vals.min(), r_vals.max())
    plt.show()

In [ ]:
# GENERACIÓN DE GRÁFICAS DE LOS MÓDULOS 1-3
# ═══════════════════════════════════════════════════════════════════════════

# Módulo 1: Simulación y visualización de dinámica newtoniana
print("Ejecutando simulación de dinámica newtoniana...")
particles = Simulacion(n_particles=2000, n_steps=1500)
plot_simulation(particles, title="Dinámica Newtoniana del Disco de Acreción")

# Módulo 2 y 3: Mapas de flujo, desplazamiento espectral y perfiles radiales
print("\nGenerando mapas de flujo observado con beaming...")
plot_flux_maps(obs_angles=(15, 45, 75))

print("\nGenerando mapas de desplazamiento espectral...")
plot_spectral_ratio_maps(obs_angles=(15, 45, 75))

print("\nGenerando mapas de desplazamiento espectral...")
plot_spectral_ratio_maps(obs_angles=(-15, -45, -75))

print("\nGenerando perfiles radiales de flujo integrado...")
plot_radial_flux_profiles(obs_angles=(15, 45, 75))

print("\n✓ Todas las gráficas han sido generadas exitosamente.")

# Teoría: Módulo 4 - Geodésicas en la Métrica de Schwarzschild

## Métrica de Schwarzschild

La métrica de Schwarzschild describe el espacio-tiempo alrededor de un agujero negro no rotante:

$$ds^2 = -\left(1 - \frac{R_s}{r}\right) dt^2 + \frac{dr^2}{1 - R_s/r} + r^2(d\theta^2 + \sin^2\theta \, d\phi^2)$$

donde $R_s = 2M$ es el radio de Schwarzschild (en unidades geométricas con $c=1$, $G=1$).

## Ecuaciones Geodésicas

Las partículas masivas siguen geodésicas determinadas por:

$$\frac{d^2x^\mu}{d\tau^2} + \Gamma^\mu_{\alpha\beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau} = 0$$

Para órbitas ecuatoriales ($\theta = \pi/2$), existen dos constantes de movimiento:
- Energía: $E = $ constante
- Momento angular: $L = r^2 \frac{d\phi}{d\tau} = $ constante

La ecuación radial en términos del parámetro afín $\tau$ es:

$$\frac{d^2r}{d\tau^2} = -\frac{M}{r^2} + \frac{L^2(r - 3M)}{r^4}$$

## Potencial Efectivo y ISCO

Se define el potencial efectivo:

$$V_{\text{eff}}(r) = \left(1 - \frac{R_s}{r}\right)\left(1 + \frac{L^2}{r^2}\right)$$

Las órbitas circulares ocurren donde $\frac{dV_{\text{eff}}}{dr} = 0$. La **órbita circular estable más interna (ISCO)** en Schwarzschild es:

$$r_{\text{ISCO}} = 6M = 3R_s$$

En el ISCO, la energía por unidad de masa es $E_{\text{ISCO}} = \frac{1}{\sqrt{3}} \approx 0.9428$.

## Precesión Orbital

A diferencia de Newton, las órbitas no son cerradas. Experimentan una precesión con ángulo por órbita:

$$\Delta\phi_{\text{prec}} \approx \frac{\pi R_s}{2a}$$

Este es un efecto relativista puro: 43 segundos de arco por siglo para Mercurio, confirmado observacionalmente.

In [ ]:
# MÓDULO 4: GEODÉSICAS EN LA MÉTRICA DE SCHWARZSCHILD
# ============================================================
# Ecuaciones de movimiento en coordenadas (r, φ) derivadas de
# la métrica ds² = -(1-Rs/r)c²dt² + dr²/(1-Rs/r) + r²dΩ²
# usando las cantidades conservadas E (energía) y L (momento angular)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# --- Parámetros (unidades geométricas: G = c = 1) ---
M  = 1.0          # masa del agujero negro
Rs = 2 * M        # radio de Schwarzschild
R_isco_s = 3 * Rs # ISCO para Schwarzschild

def geodesica_schwarzschild(r0, v_r0, L, M=1.0, tau_max=500, n=50000):
    """
    Integra la geodésica de una partícula masiva en Schwarzschild.
    
    Estado: [r, phi, dr/dtau]
    Constante conservada: L = r² dphi/dtau  (momento angular)
    
    Ecuación radial (de la métrica):
        (dr/dtau)² = E² - (1 - Rs/r)(1 + L²/r²)
    
    En forma de sistema de primer orden:
        dr/dtau  = u
        dphi/dtau = L / r²
        du/dtau  = -M/r² + L²*(r - 3M)/r⁴   ← potencial efectivo
    """
    Rs_loc = 2 * M

    def eqs(tau, state):
        r, phi, u = state
        if r <= Rs_loc * 1.001:          # absorbido por el horizonte
            return [0, 0, 0]
        dphi_dtau = L / r**2
        du_dtau   = -M / r**2 + L**2 * (r - 3*M) / r**4
        return [u, dphi_dtau, du_dtau]

    def hit_horizon(tau, state):
        return state[0] - Rs_loc * 1.001
    hit_horizon.terminal  = True
    hit_horizon.direction = -1

    sol = solve_ivp(
        eqs,
        [0, tau_max],
        [r0, 0.0, v_r0],
        max_step=tau_max / n,
        events=hit_horizon,
        rtol=1e-10, atol=1e-13
    )
    r   = sol.y[0]
    phi = sol.y[1]
    return r * np.cos(phi), r * np.sin(phi), r, phi


def potencial_efectivo(r_arr, L, M=1.0):
    """
    V_eff(r) = (1 - Rs/r)(1 + L²/r²)
    La partícula puede orbitar donde V_eff = E².
    """
    Rs_loc = 2 * M
    return (1 - Rs_loc / r_arr) * (1 + L**2 / r_arr**2)


# --- Ejemplo: comparación Newton vs Schwarzschild ---
def orbita_newtoniana(r0, L, M=1.0, t_max=600, n=50000):
    """Órbita newtoniana equivalente para comparar."""
    def eqs(t, state):
        r, phi, u = state
        dphi_dt = L / r**2
        du_dt   = -M / r**2 + L**2 / r**3   # sin término -3M en r⁴
        return [u, dphi_dt, du_dt]
    sol = solve_ivp(eqs, [0, t_max], [r0, 0.0, 0.0],
                    max_step=t_max/n, rtol=1e-10, atol=1e-13)
    r, phi = sol.y[0], sol.y[1]
    return r * np.cos(phi), r * np.sin(phi)


# ---- Visualización ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#08090d')
for ax in axes:
    ax.set_facecolor('#08090d')
    ax.tick_params(colors='#aaaacc')

# Panel 1: potencial efectivo para distintos L
ax = axes[0]
r_plot = np.linspace(Rs * 1.05, 20 * Rs, 800)
for L_val, col in [(3.5, '#ffd07a'), (4.0, '#ff8c42'), (5.0, '#80d4ff')]:
    V = potencial_efectivo(r_plot, L_val)
    ax.plot(r_plot / Rs, V, color=col, label=f'L={L_val}')
ax.axvline(1, color='red', lw=1, linestyle='--', label='Horizonte')
ax.axvline(3, color='orange', lw=0.8, linestyle=':', label='ISCO (3Rₛ)')
ax.set_xlim(0.8, 12); ax.set_ylim(0.7, 1.4)
ax.set_xlabel('r / Rₛ', color='#aaaacc')
ax.set_ylabel('V_eff(r)', color='#aaaacc')
ax.set_title('Potencial efectivo Schwarzschild', color='#f0c060')
ax.legend(fontsize=8)

# Panel 2: órbitas Schwarzschild (distintas L)
ax = axes[1]
theta_circ = np.linspace(0, 2*np.pi, 300)
ax.fill(Rs * np.cos(theta_circ), Rs * np.sin(theta_circ), color='black')
ax.plot(Rs * np.cos(theta_circ), Rs * np.sin(theta_circ), 'r-', lw=1)
L_valores = [3.5, 4.0, 4.5, 5.0]
colores    = ['#ffd07a', '#ff8c42', '#80d4ff', '#a0ffa0']
for L_val, col in zip(L_valores, colores):
    x, y, r, _ = geodesica_schwarzschild(8*Rs, 0.0, L_val, tau_max=800)
    ax.plot(x/Rs, y/Rs, color=col, lw=0.8, label=f'L={L_val}')
ax.set_xlim(-15, 15); ax.set_ylim(-15, 15); ax.set_aspect('equal')
ax.set_title('Geodésicas Schwarzschild', color='#f0c060')
ax.legend(fontsize=8)

# Panel 3: precesión — comparar Newton vs Schwarzschild
ax = axes[2]
L_prec = 4.3
x_s, y_s, _, _ = geodesica_schwarzschild(7*Rs, 0.0, L_prec, tau_max=3000)
x_n, y_n       = orbita_newtoniana(7*Rs, L_prec, t_max=3000)
ax.fill(Rs * np.cos(theta_circ), Rs * np.sin(theta_circ), color='black')
ax.plot(Rs * np.cos(theta_circ), Rs * np.sin(theta_circ), 'r-', lw=1)
ax.plot(x_n/Rs, y_n/Rs, color='#80d4ff', lw=0.7, alpha=0.6, label='Newton')
ax.plot(x_s/Rs, y_s/Rs, color='#ffd07a', lw=0.7, alpha=0.8, label='Schwarzschild')
ax.set_xlim(-12, 12); ax.set_ylim(-12, 12); ax.set_aspect('equal')
ax.set_title('Precesión orbital (L=4.3)', color='#f0c060')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('schwarzschild_geodesicas.png', dpi=150, bbox_inches='tight')
plt.show()

# Teoría: Módulo 5 - Trayectorias de Fotones y Sombra del Agujero Negro

## Geodésicas Nulas

Los fotones siguen geodésicas nulas ($ds^2 = 0$). Normalizando $E = 1$ y definiendo el parámetro de impacto $b = L/E = L$, la ecuación radial queda:

$$\left(\frac{dr}{d\lambda}\right)^2 = 1 - \frac{b^2}{r^2}\left(1 - \frac{R_s}{r}\right)$$

donde $\lambda$ es el parámetro afín. La aceleración radial se obtiene diferenciando esta restricción:

$$\frac{d^2r}{d\lambda^2} = \frac{b^2(r - 3M)}{r^4}$$

## Esfera de Fotones

Existe una órbita circular inestable de fotones en:

$$r_{\text{fotones}} = 3M = \frac{3R_s}{2}$$

Esta órbita es inestable: cualquier perturbación pequeña causa que el fotón escape o sea capturado por el horizonte.

## Parámetro de Impacto Crítico

El valor crítico por debajo del cual los fotones son capturados es:

$$b_{\text{crit}} = 3\sqrt{3}\,M = \frac{3\sqrt{3}}{2}R_s \approx 5.196\,M$$

- Si $b < b_{\text{crit}}$: fotón capturado (rojo en visualización)
- Si $b > b_{\text{crit}}$: fotón deflectado (amarillo en visualización)

## La Sombra del Agujero Negro

La "sombra" es la proyección de la región de captura de fotones. Su radio angular observado es:

$$\theta_{\text{shadow}} \approx \frac{b_{\text{crit}}}{D_{\text{obs}}}$$

Para un observador a distancia infinita, define el tamaño angular del agujero negro como se observa desde la Tierra (importante para Event Horizon Telescope).

In [ ]:
# ============================================================
# MÓDULO 5: TRAYECTORIAS DE FOTONES Y SOMBRA DEL AGUJERO NEGRO
# ============================================================
# Para fotones (masa = 0), la geodésica nula en Schwarzschild
# se parametriza por el parámetro de impacto b = L/E.
# La esfera de fotones inestable está en r = 3M (= 1.5 Rₛ).
# Fotones con b < b_crit = 3√3 M son capturados.
# ============================================================

def geodesica_foton(b, r0=30.0, M=1.0, dlambda=0.04, n_steps=30000):
    """
    Integra la geodésica nula de un fotón con parámetro de impacto b.

    Normalización E = 1  (L = b):
        (dr/dλ)²  = 1 - b²(1 - 2M/r)/r²   [restricción nula]
        dφ/dλ     = b / r²                  [momento angular conservado]
        d²r/dλ²   = b²(r - 3M) / r⁴        [geodésica nula exacta]
    """
    Rs_loc = 2 * M

    # Ángulo inicial: fotón viene desde la derecha (φ≈0) con offset perpendicular b
    phi0 = np.arcsin(np.clip(b / r0, -1.0, 1.0))

    def eqs(lam, state):
        r, phi, u = state
        if r <= Rs_loc * 1.005:
            return [0, 0, 0]
        dphi_dlam = b / r**2
        du_dlam   = b**2 * (r - 3*M) / r**4   # aceleración radial exacta
        return [u, dphi_dlam, du_dlam]

    def evento_captura(lam, state):
        return state[0] - Rs_loc * 1.005
    evento_captura.terminal  = True
    evento_captura.direction = -1

    def evento_escape(lam, state):
        return state[0] - r0 * 1.2
    evento_escape.terminal  = True
    evento_escape.direction = 1   # solo se activa cuando r está aumentando

    # Condición inicial correcta con E=1: fotón moviéndose hacia el BH
    u0 = -np.sqrt(max(1.0 - b**2 * (1.0 - Rs_loc / r0) / r0**2, 0.0))

    sol = solve_ivp(
        eqs,
        [0, n_steps * dlambda],
        [r0, phi0, u0],
        max_step=dlambda,
        events=[evento_captura, evento_escape],
        rtol=1e-10, atol=1e-13
    )
    r, phi = sol.y[0], sol.y[1]
    capturado_flag = len(sol.t_events[0]) > 0
    return r * np.cos(phi), r * np.sin(phi), capturado_flag


# --- Calcular sombra y anillo de fotones ---
M_val  = 1.0
b_crit = 3 * np.sqrt(3) * M_val   # ≈ 5.196

b_values = np.linspace(b_crit * 0.5, b_crit * 2.5, 150)

fig, ax = plt.subplots(figsize=(10, 10), facecolor='#04050a')
ax.set_facecolor('#04050a')

# Círculo del horizonte
theta_c = np.linspace(0, 2*np.pi, 300)
ax.fill(2*M_val * np.cos(theta_c), 2*M_val * np.sin(theta_c), color='black', zorder=5)

# Esfera de fotones
ax.plot(3*M_val * np.cos(theta_c), 3*M_val * np.sin(theta_c),
        '--', color='orange', lw=0.8, alpha=0.5, label='Esfera de fotones (r=3M)')

for b in b_values:
    x, y, cap = geodesica_foton(b, r0=26*M_val, M=M_val)
    color = '#ff4400' if cap else '#ffe070'
    alpha = 0.35 if cap else 0.65
    ax.plot(x,  y, color=color, lw=0.5, alpha=alpha)
    ax.plot(x, -y, color=color, lw=0.5, alpha=alpha)  # simetría b → -b

ax.set_xlim(-28, 28); ax.set_ylim(-28, 28); ax.set_aspect('equal')
ax.set_title('Sombra del agujero negro — fotones en Schwarzschild\n'
             'Rojo = capturado | Amarillo = deflectado',
             color='#f0c060', fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.tick_params(colors='#aaaacc')
plt.tight_layout()
plt.savefig('shadow_schwarzschild.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Parámetro de impacto crítico b_crit = 3√3 M = {b_crit:.4f}")
print(f"Ángulo de la sombra observado: θ_shadow ≈ b_crit / D_obs")

# Teoría: Módulo 6 - Geodésicas en la Métrica de Kerr

## Métrica de Kerr

La métrica de Kerr describe el espacio-tiempo alrededor de un agujero negro **rotante** en coordenadas de Boyer-Lindquist:

$$ds^2 = -\frac{\Delta - a^2\sin^2\theta}{\Sigma} dt^2 + \frac{\Sigma}{\Delta} dr^2 + \Sigma \, d\theta^2 + \frac{\sin^2\theta}{\Sigma}[(r^2+a^2)^2 - \Delta a^2\sin^2\theta] d\phi^2$$

donde:

$$\Sigma = r^2 + a^2\cos^2\theta, \quad \Delta = r^2 - 2Mr + a^2$$

El parámetro $a = J/M$ es el **espín específico**, con rango $0 \le a \le M$.

## Cantidades Conservadas

A diferencia de Schwarzschild, en Kerr hay tres cantidades conservadas:
- Energía: $E$
- Momento angular axial: $L$
- Constante de Carter: $Q$ (relacionada con separabilidad)

## ISCO en Kerr

Para órbitas ecuatoriales pro-grado, el ISCO está en:

$$r_{\text{ISCO}}^+(a) = M\left(3 + Z_2 - \sqrt{(3-Z_1)(3+Z_1+2Z_2)}\right)$$

donde:

$$Z_1 = 1 + \left(1-\frac{a^2}{M^2}\right)^{1/3}\left[\left(1+\frac{a}{M}\right)^{1/3} + \left(1-\frac{a}{M}\right)^{1/3}\right]$$

$$Z_2 = \sqrt{3\frac{a^2}{M^2} + Z_1^2}$$

**Casos límite:**
- $a = 0$ (Schwarzschild): $r_{\text{ISCO}} = 6M$
- $a = M$ (extremo): $r_{\text{ISCO}} = M$ (pro-grado), $r_{\text{ISCO}} = 9M$ (retro-grado)

## Arrastre de Marco

Un efecto puramente Kerr es el **arrastre de marco**: el espacio-tiempo alrededor del agujero negro rotante está siendo "arrastra". La velocidad de arrastre es:

$$\Omega_{\text{drag}} = \frac{2aMr}{(Mr\Sigma - \Delta)\Delta}$$

## Eficiencia de Acreción

La eficiencia radiativa depende de la energía en el ISCO:

$$\eta = 1 - E_{\text{ISCO}}$$

**Comparación:**
- Schwarzschild: $\eta \approx 5.7\%$
- Kerr $a=0.5M$: $\eta \approx 13.7\%$
- Kerr $a=0.9M$: $\eta \approx 28.7\%$
- Kerr extremo $a=M$: $\eta \approx 42.3\%$

Los agujeros negros rotantes son **7 veces más luminosos**!

In [ ]:
# ============================================================
# MÓDULO 6: GEODÉSICAS EN LA MÉTRICA DE KERR
# ============================================================
# La métrica de Kerr en coordenadas de Boyer-Lindquist:
#   Σ = r² + a²cos²θ,   Δ = r² - 2Mr + a²
#   a = J/M  (parámetro de spin, 0 ≤ a ≤ M)
#
# Cantidades conservadas: E (energía), L (momento angular axial), Q (Carter)
# Para órbitas ecuatoriales (θ=π/2, Q=0):
#   ISCO pro-grado: r = M(3 + Z₂ - √((3-Z₁)(3+Z₁+2Z₂)))
#   ISCO retro-grado: r = M(3 + Z₂ + √((3-Z₁)(3+Z₁+2Z₂)))
# ============================================================

def isco_kerr(a, M=1.0):
    """
    Radio del ISCO en métrica de Kerr para órbitas ecuatoriales.
    a: parámetro de spin (0 ≤ a ≤ M)
    Retorna (r_isco_pro, r_isco_retro).
    """
    z1 = 1 + (1 - a**2/M**2)**(1/3) * (
         (1 + a/M)**(1/3) + (1 - a/M)**(1/3))
    z2 = np.sqrt(3 * a**2 / M**2 + z1**2)
    r_pro  = M * (3 + z2 - np.sqrt((3 - z1)*(3 + z1 + 2*z2)))
    r_retro= M * (3 + z2 + np.sqrt((3 - z1)*(3 + z1 + 2*z2)))
    return r_pro, r_retro


def geodesica_kerr_ecuatorial(r0, L, a, M=1.0, E=0.95, tau_max=1000, n=60000):
    """
    Integra geodésica ecuatorial (θ=π/2) en métrica de Kerr.
    Variables: [r, phi, u_r]  donde u_r = dr/dτ
    
    Ecuación radial efectiva:
        R(r) = [E(r²+a²) - La]² - Δ[r² + (L-aE)²]
        (dr/dτ)² = R(r) / r^4
    """
    def delta(r):
        return r**2 - 2*M*r + a**2

    def R_radial(r):
        D = delta(r)
        return (E*(r**2 + a**2) - L*a)**2 - D*(r**2 + (L - a*E)**2)

    def dR_dr(r):
        D  = delta(r)
        dD = 2*r - 2*M
        term1 = 2*(E*(r**2+a**2) - L*a) * (2*E*r)
        term2 = dD*(r**2 + (L - a*E)**2) + D*2*r
        return term1 - term2

    def eqs(tau, state):
        r, phi, u = state
        r_h = M + np.sqrt(M**2 - a**2)   # horizonte exterior
        if r <= r_h * 1.001:
            return [0, 0, 0]
        D = delta(r)
        # dphi/dtau (arrastre de marco en Kerr)
        dphi = (a*(E*(r**2+a**2) - L*a)/D - a*E + L) / r**2
        # du/dtau = ½ dR/dr / r^4  (aprox. para θ=π/2)
        du   = 0.5 * dR_dr(r) / r**4
        return [u, dphi, du]

    # Condición inicial: partícula en órbita aproximada
    R0  = R_radial(r0)
    u0  = np.sqrt(max(R0, 0)) / r0**2

    sol = solve_ivp(
        eqs, [0, tau_max],
        [r0, 0.0, -u0],       # -u0: moviendose hacia el BH
        max_step=tau_max/n,
        rtol=1e-10, atol=1e-13
    )
    r, phi = sol.y[0], sol.y[1]
    return r * np.cos(phi), r * np.sin(phi), r


# --- Comparación: ISCO vs parámetro de spin ---
a_vals   = np.linspace(0, 0.99, 200)
isco_pro = np.array([isco_kerr(a)[0] for a in a_vals])
isco_ret = np.array([isco_kerr(a)[1] for a in a_vals])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#08090d')
for ax in axes:
    ax.set_facecolor('#08090d'); ax.tick_params(colors='#aaaacc')

# Panel 1: ISCO vs spin
ax = axes[0]
ax.plot(a_vals, isco_pro,  color='#ffd07a', lw=2, label='ISCO pro-grado')
ax.plot(a_vals, isco_ret,  color='#80d4ff', lw=2, label='ISCO retro-grado')
ax.axhline(6, color='gray', lw=1, linestyle='--', alpha=0.5, label='Schwarzschild (a=0)')
ax.fill_between(a_vals, 2*a_vals, isco_pro, alpha=0.1, color='red',
                label='Región dentro del horizonte')
ax.set_xlabel('Parámetro de spin a/M', color='#aaaacc')
ax.set_ylabel('r_ISCO / M', color='#aaaacc')
ax.set_title('Radio ISCO en métrica de Kerr', color='#f0c060')
ax.legend(fontsize=9)
ax.set_ylim(0, 10)

# Panel 2: órbitas para distintos spins
ax = axes[1]
theta_c = np.linspace(0, 2*np.pi, 300)
ax.fill(np.cos(theta_c), np.sin(theta_c), color='black')   # BH (unidades M=1)
ax.plot(np.cos(theta_c), np.sin(theta_c), 'r-', lw=1)

for a_val, col, label in [(0.0, '#80d4ff', 'a=0 (Schwarzschild)'),
                           (0.5, '#ffd07a', 'a=0.5 M'),
                           (0.9, '#ff8c42', 'a=0.9 M')]:
    r_i, _ = isco_kerr(a_val)
    L_orb  = np.sqrt(r_i) * (r_i**2 - 2*a_val*np.sqrt(r_i) + a_val**2) / \
             (r_i**(3/4) * np.sqrt(r_i**2 - 3*r_i + 2*a_val*np.sqrt(r_i)))
    x, y, r = geodesica_kerr_ecuatorial(r_i * 2.5, L_orb, a=a_val, tau_max=2000)
    ax.plot(x, y, color=col, lw=0.8, alpha=0.85, label=label)

ax.set_xlim(-15, 15); ax.set_ylim(-15, 15); ax.set_aspect('equal')
ax.set_title('Geodésicas ecuatoriales — distintos spins', color='#f0c060')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('kerr_geodesicas.png', dpi=150, bbox_inches='tight')
plt.show()

# Teoría: Módulo 7 - Síntesis Comparativa y Mapas de Emisión

## Integración de Efectos: Mapa de Disco Observado

El flujo observado en cada punto del disco es una combinación de tres efectos relativistas:

$$I_{\text{obs}}(r, \phi) = \frac{T(r)^4}{\delta(r,\phi)^3} \cdot \sqrt{1 - \frac{R_s}{r}}$$

donde:

- **$T(r)^4$**: Radiación térmica (Stefan-Boltzmann con perfil Shakura-Sunyaev)
- **$\delta^{-3}(r,\phi)$**: Beaming relativista ($\delta = \gamma(1-\beta\cos\theta)$)
- **$\sqrt{1-R_s/r}$**: Redshift gravitacional

## Geometría de Inclinación

Para un observador a inclinación $i$ del eje del disco:

$$\cos\theta(r,\phi) = \sin(i) \sin(\phi)$$

- $i = 0°$: Cara frontal (débil asimetría)
- $i = 45°$: Inclinación intermedia (asimetría creciente)
- $i = 85°$: Casi de canto (máxima asimetría, fuerte "hot spot")

## Longitud de Onda Observada

Cada fotón experimenta dos corrimientos:

$$\lambda_{\text{obs}} = \lambda_0 \cdot \underbrace{\gamma(1-\beta\cos\theta)}_{\text{Doppler}} \cdot \underbrace{\frac{1}{\sqrt{1-R_s/r}}}_{\text{Redshift gravitacional}}$$

## Espectro Integrado

El espectro total observado es:

$$F(\lambda, i) = \int_{\text{disco}} \frac{B_\lambda(T(r))}{\delta(r,\phi,i)^3} \sqrt{1 - \frac{R_s}{r}} \, dA$$

donde $dA = r \, dr \, d\phi$ es el elemento de área.

La inclinación del disco es observable a través de:
1. **Asimetría espectral**: lado azul (próximo) vs. rojo (lejano)
2. **Forma del perfil radial**: amplificación del beaming
3. **Ancho de línea**: $\Delta\lambda \propto v_{\text{orb}}$ en el ISCO

## Implicaciones Astrofísicas

Los mapas de emisión permiten:

- **Caracterizar agujeros negros**: Inferir $M$ y $a$ desde observaciones
- **Comprender AGN**: Explicar variabilidad rápida y espectros duros
- **Binarias de rayos X**: Modelar explosiones de tipo I desde discos de acreción
- **Event Horizon Telescope**: Interpretar imágenes de la sombra + anillo de fotones

In [ ]:
# ============================================================
# MÓDULO 7: SÍNTESIS COMPARATIVA
# ============================================================

import pandas as pd

def eficiencia_acrecion(a, M=1.0):
    """
    Eficiencia radiativa η = 1 - √(1 - 2M/(3 r_isco))
    (para geodésica circular en Schwarzschild/Kerr)
    """
    r_i, _ = isco_kerr(a, M)
    E_isco = (1 - 2*M/(3*r_i))**0.5
    return 1 - E_isco

# Tabla comparativa
datos = []
for a_val in [0.0, 0.5, 0.9, 0.99]:
    r_pro, r_ret = isco_kerr(a_val)
    eta = eficiencia_acrecion(a_val)
    datos.append({
        'Spin a/M'          : a_val,
        'ISCO pro-grado (M)': round(r_pro, 3),
        'ISCO retro (M)'    : round(r_ret, 3),
        'η acreción (%)'    : round(eta * 100, 2),
        'b_crit fotones'    : round(3*np.sqrt(3)*(1 - a_val*0.3), 3),  # aprox
    })

df = pd.DataFrame(datos)
print("\n=== Tabla comparativa: Schwarzschild vs Kerr ===")
print(df.to_string(index=False))

# --- Perfil espectral integrado del disco ---
# Combina temperatura Shakura-Sunyaev + corrimientos relativistas
# para producir el espectro observado F(λ) desde distintos ángulos

from scipy.constants import h, c as c_SI, k  # constantes de Planck

def planck_nm(lam_nm, T):
    """Intensidad de cuerpo negro en función de λ (nm) y T (K)."""
    lam_m = lam_nm * 1e-9
    return (2*h*c_SI**2 / lam_m**5) / (np.exp(h*c_SI/(lam_m*k*T)) - 1)

def espectro_disco(R_s, M_BH_kg, G_val, inclinacion_deg=30,
                   N_r=100, N_phi=200, lam_arr=None):
    """
    Integra el espectro observado del disco sumando la contribución
    de cada anillo, corregido por:
      - redshift gravitacional Schwarzschild
      - beaming + Doppler relativista
    """
    if lam_arr is None:
        lam_arr = np.linspace(0.1, 1000, 2000)   # nm (UV a IR)

    inc = np.radians(inclinacion_deg)
    G   = 6.674e-11
    c_v = 3e8

    R_isco_v = 3 * R_s
    r_arr    = np.linspace(R_isco_v*1.05, 20*R_s, N_r)
    phi_arr  = np.linspace(0, 2*np.pi, N_phi, endpoint=False)
    PHI, R   = np.meshgrid(phi_arr, r_arr)

    v_kep    = np.sqrt(G * M_BH_kg / R)
    beta     = v_kep / c_v
    gamma    = 1 / np.sqrt(1 - beta**2)
    cos_th   = beta * np.sin(PHI) * np.sin(inc) / beta   # simplificado

    grav_fac = np.sqrt(np.clip(1 - R_s/R, 1e-6, 1))     # redshift gravitacional
    doppler  = gamma * (1 - beta * cos_th)
    T_local  = 1e7 * (R/R_isco_v)**(-3/4) * np.maximum(1 - np.sqrt(R_isco_v/R), 0)**0.25

    F_total = np.zeros_like(lam_arr)
    for i, lam in enumerate(lam_arr):
        lam_emit = lam * grav_fac * doppler            # λ emitido corregido
        lam_emit = np.clip(lam_emit, 0.01, 5000)
        B        = planck_nm(lam_emit, np.clip(T_local, 1, 1e9))
        F_obs    = B / doppler**3                       # beaming
        F_total[i] = np.nansum(F_obs[T_local > 0])

    return lam_arr, F_total / np.max(F_total)


# Ejemplo de uso
print("\nCalculando espectros para distintas inclinaciones...")
fig, ax = plt.subplots(figsize=(10, 5), facecolor='#08090d')
ax.set_facecolor('#08090d'); ax.tick_params(colors='#aaaacc')

G_SI   = 6.674e-11
M_BH_k = 10 * 1.989e30
R_s_m  = 2 * G_SI * M_BH_k / (3e8)**2

lam = np.linspace(1, 800, 600)
for ang, col in [(10, '#80d4ff'), (45, '#ffd07a'), (75, '#ff8c42'), (85, '#ff4444')]:
    _, F = espectro_disco(R_s_m, M_BH_k, G_SI, inclinacion_deg=ang, lam_arr=lam)
    ax.plot(lam, F, color=col, lw=1.5, label=f'i = {ang}°')

ax.set_xlabel('Longitud de onda (nm)', color='#aaaacc')
ax.set_ylabel('Flujo normalizado', color='#aaaacc')
ax.set_title('Espectro observado del disco — efectos de inclinación + relatividad',
             color='#f0c060')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('espectro_disco.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen: Ecuaciones Fundamentales por Módulo

## 🟢 Módulo 1: Dinámica Newtoniana

| Cantidad | Fórmula | Significado |
|----------|---------|-------------|
| Velocidad circular | $v_c = \sqrt{GM/r}$ | Balance gravitacional-centrífugo |
| Período orbital | $T = 2\pi\sqrt{r^3/GM}$ | Tercera ley de Kepler |
| Aceleración gravitacional | $a = -GM/r^2$ | Fuerza por unidad de masa |

## 🟠 Módulo 2: Relatividad Especial

| Cantidad | Fórmula | Significado |
|----------|---------|-------------|
| Factor de Lorentz | $\gamma = 1/\sqrt{1-\beta^2}$ | Dilatación temporal |
| Factor Doppler | $\delta = \gamma(1-\beta\cos\theta)$ | Efecto combinado Doppler + Lorentz |
| Beaming | $F_{\text{obs}} = F_0/\delta^3$ | Amplificación por movimiento relativista |
| Desplazamiento Doppler | $\lambda_{\text{obs}} = \lambda_0 \delta$ | Corrimiento de longitud de onda |

## 🔵 Módulo 3: Radiación Térmica

| Cantidad | Fórmula | Significado |
|----------|---------|-------------|
| Temperatura (Shakura-Sunyaev) | $T(r) = T_0(R_i/r)^{3/4}(1-\sqrt{R_i/r})^{1/4}$ | Perfil de disco en acreción |
| Planck | $B_\lambda = 2hc^2/[\lambda^5(e^{hc/\lambda k_B T}-1)]$ | Radiación de cuerpo negro |
| Ley de Wien | $\lambda_{\text{pico}} = b/T$ | Pico de emisión térmica |
| Redshift gravitacional | $\lambda = \lambda_0/\sqrt{1-R_s/r}$ | Corrimiento por gravedad |

## ⚫ Módulos 4, 5, 6: Relatividad General

| Concepto | Schwarzschild | Kerr (Rotante) |
|----------|---------------|----------------|
| ISCO | $r = 6M$ | $r(a)$ con $M$ a $9M$ |
| Esfera fotones | $r = 3M/2$ | $r = 3M/2$ (aprox.) |
| Parámetro impacto crítico | $b_c = 3\sqrt{3}M$ | $b_c(a)$ varía poco |
| Eficiencia acreción | $\eta = 5.7\%$ | $\eta = 42.3\%$ (máximo) |
| Efecto especial | Precesión | Arrastre de marco |

## 📊 Resumen de Unidades

**Sistema empleado: Unidades Geométricas**

| Variable | Valor | Conversión |
|----------|-------|-----------|
| Velocidad luz | $c = 1$ | $3 \times 10^8$ m/s |
| Constante gravedad | $G = 1$ | $6.674 \times 10^{-11}$ m³/(kg·s²) |
| Masa agujero negro | $M_{\text{BH}} = 10 M_\odot$ | $1.989 \times 10^{31}$ kg |
| Radio Schwarzschild | $R_s = 2M$ | $\sim 30$ km para 10 $M_\odot$ |
| ISCO | $R_{\text{ISCO}} = 3R_s$ | $\sim 90$ km |